# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, their `@id`s, and columns. This helps identify what data is available for extraction and processing.

In [ ]:
# List all record sets and their fields
print("Available Record Sets (by @id):")
for record_set in metadata.record_sets:
    print(f"- {record_set['@id']}: {record_set.get('name', 'N/A')}")
    print("  Fields and their @id's:")
    for field in record_set.get('fields', []):
        print(f"    - {field['@id']} (name={field.get('name', 'N/A')})")
    print("  Columns and their @id's (if present):")
    for column in record_set.get('columns', []):
        print(f"    - {column['@id']} (name={column.get('name', 'N/A')})")
    print()

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis. Use record set and field `@id`s from the overview.

In [ ]:
import pprint

# List the record set @id's found above (replace with actual available @id's):
record_set_ids = [rs['@id'] for rs in metadata.record_sets]

dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading data for RecordSet: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Columns for {record_set_id}:", df.columns.tolist())
            display(df.head())
        else:
            print(f"No records found for {record_set_id}.")
    except Exception as e:
        print(f"Error loading {record_set_id}: {e}")

pprint.pprint({rs: list(df.columns) for rs, df in dataframes.items()})

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes filtering, normalization, and grouping using the `@id` for fields.

In [ ]:
# Example: Assume we pick the first non-empty record set and a numeric field for analysis

# You may have to adjust these manually after running the above code to see available IDs.
if len(dataframes) > 0:
    # Pick the first non-empty DataFrame
    first_rs_id = next(iter(dataframes))
    df = dataframes[first_rs_id]
    print(f"Working with record set: {first_rs_id}")

    # Try to detect a numeric field automatically for demo purposes
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
        # Try to coerce
        try:
            df[col] = pd.to_numeric(df[col], errors='ignore')
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break
        except:
            continue

    if numeric_field_id:
        print(f"Using numeric field {numeric_field_id} for EDA.")

        threshold = df[numeric_field_id].mean() if not pd.isna(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        filtered_df = filtered_df.copy()  # To avoid SettingWithCopyWarnings
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to group by a likely categorical/text field
        group_field = None
        for col in df.columns:
            if col != numeric_field_id and df[col].dtype == object:
                group_field = col
                break

        if group_field:
            print(f"Grouping by {group_field} (field @id: {group_field})")
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            display(grouped_df.head())
    else:
        print("No numeric field found for demonstration.")
else:
    print("No record sets with tabular data available.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here's an example of plotting a numeric field's distribution and group-wise means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) > 0 and 'numeric_field_id' in locals() and numeric_field_id:
    df = dataframes[first_rs_id]
    if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
        plt.figure(figsize=(8, 4))
        sns.histplot(df[numeric_field_id].dropna(), kde=True)
        plt.title(f'Distribution of {numeric_field_id} (@id: {numeric_field_id})')
        plt.xlabel(numeric_field_id)
        plt.ylabel('Frequency')
        plt.tight_layout()
        plt.show()

        if 'group_field' in locals() and group_field and group_field in df.columns:
            group_means = df.groupby(group_field)[numeric_field_id].mean().sort_values(ascending=False)
            plt.figure(figsize=(10,5))
            sns.barplot(x=group_means.values, y=group_means.index)
            plt.title(f'Mean {numeric_field_id} by {group_field}')
            plt.xlabel(f'Mean {numeric_field_id}')
            plt.ylabel(group_field)
            plt.tight_layout()
            plt.show()
    else:
        print(f"Field {numeric_field_id} does not appear to be numeric for plotting.")
else:
    print("No data available for visualization.")

## 6. Conclusion
This notebook demonstrated how to load and explore the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the `mlcroissant` library. We inspected the available record sets, loaded tabular data, performed basic exploratory data analysis, and visualized distributions and group-wise metrics. 

**Key Observations:**
- The names and organization of fields/columns are provided by the Croissant schema `@id`, which is crucial for consistent referencing and downstream FAIR data operations.
- The dataset supports straightforward programmatic data access, filtering, and analysis using the Croissant metadata.

Further steps might include more advanced analyses, feature engineering, or combining this resource with domain knowledge for downstream machine learning tasks.